# 06 - Pipeline Validation (Bronze, Silver, Gold)

All testing and validation checks of the medallion pipeline in one place. Runs as the last job task, after `gold_analytics`. The layer notebooks (01-05) only build data.

The notebook is read-only for Bronze, Silver and Gold data. Its only writes are one summary table per layer (overwritten each run):
- `capstone6_telecom.bronze.bronze_validation_summary`
- `capstone6_telecom.silver.silver_validation_summary`
- `capstone6_telecom.gold.gold_validation_summary`

Each row is one check: `layer`, `check_group`, `check_name`, `table_name`, `comparison`, `expected_value`, `actual_value`, `blocking`, `status` (PASS / FAIL / WARN), `validated_at`.
- Blocking checks test pipeline correctness (reconciliation, uniqueness, rule results, Gold totals). A failure is `FAIL`, and the task fails, so the job's failure email is sent.
- Non-blocking checks report source-data consistency (region checks). A failure is `WARN` and does not fail the job.

| Layer | Checks |
|---|---|
| Bronze | Tables not empty, `_ingested_at` populated, profiling summary = current row counts, region consistency on raw values (latest version per master key) |
| Silver | Bronze = Silver + rejected, unique non-null keys, warning flags = warning table, rule catalogue integrity, rule reconciliation (Bronze validation query = detected = stored), region consistency |
| Gold | Facts/dimensions = Silver, one row per grain, daily/SLA totals = facts, independent latency and impact-window recomputation, rejected records absent, region flag accuracy |

No `dbutils` is used.

In [0]:
# 1. Configuration and check helpers
from pyspark.sql import functions as F

catalog = "capstone6_telecom"
B, S, G = f"{catalog}.bronze", f"{catalog}.silver", f"{catalog}.gold"

FAIL_ON_ERROR = True   # raise at the end when a blocking check fails, so the job task fails and the failure email is sent

SUMMARY_TABLES = {"BRONZE": f"{B}.bronze_validation_summary",
                  "SILVER": f"{S}.silver_validation_summary",
                  "GOLD": f"{G}.gold_validation_summary"}

results = []   # one tuple per check

def check(layer, group, name, table, expected, actual, op="=", blocking=True):
    """Record one check. op '=' -> actual == expected; op '>=' -> actual >= expected.
    A failed blocking check is FAIL; a failed non-blocking check is WARN."""
    expected, actual = int(expected or 0), int(actual or 0)
    ok = actual == expected if op == "=" else actual >= expected
    status = "PASS" if ok else ("FAIL" if blocking else "WARN")
    results.append((layer, group, name, table, op, expected, actual, blocking, status))

def q(sql):
    """First row of a SQL query."""
    return spark.sql(sql).first()

def v(c, a="t"):
    """Trimmed text value of a column, blank -> NULL (same standardisation as Silver)."""
    return f"NULLIF(TRIM(CAST({a}.`{c}` AS STRING)), '')"

def total(df, c):
    return df.agg(F.coalesce(F.sum(c), F.lit(0))).first()[0]

# Bronze table -> (Silver table, business key)
DATASETS = {
    "bronze_region_lookup": ("silver_region_lookup", "region_id"),
    "bronze_tower_master": ("silver_tower_master", "tower_id"),
    "bronze_network_topology_master": ("silver_network_topology_master", "cell_id"),
    "bronze_network_kpi_measurements": ("silver_network_kpi_measurements", "measurement_id"),
    "bronze_network_outage_events": ("silver_network_outage_events", "outage_id"),
    "bronze_customer_complaints": ("silver_customer_complaints", "complaint_id"),
}

In [0]:
# 2. Bronze checks (moved from 03_Bronze_Data_Profiling)
profile_rows = {r.table_name: r.total_rows for r in
                spark.table(f"{B}.bronze_profiling_summary").filter("column_name = 'ALL_COLUMNS'").collect()}

for t in DATASETS:
    r = q(f"SELECT COUNT(*) AS n, COUNT_IF(_ingested_at IS NULL) AS no_ts FROM {B}.{t}")
    check("BRONZE", "VOLUME", "Table is not empty", t, 1, r.n, ">=")
    check("BRONZE", "LINEAGE", "_ingested_at populated (rows missing)", t, 0, r.no_ts)
    check("BRONZE", "PROFILING", "bronze_profiling_summary total_rows = current row count", t, r.n, profile_rows.get(t, -1))

# Region consistency on the raw Bronze values (non-blocking: source-data issue, records are never dropped for it).
# Masters can hold several loaded versions of a key, so the reference is the latest loaded version per key.
spark.sql(f"""CREATE OR REPLACE TEMP VIEW v_tower_ref AS
    SELECT {v('tower_id')} AS tower_id, max_by({v('region_id')}, t._ingested_at) AS region_id
    FROM {B}.bronze_tower_master t GROUP BY 1""")
spark.sql(f"""CREATE OR REPLACE TEMP VIEW v_cell_ref AS
    SELECT {v('cell_id')} AS cell_id, max_by({v('region_id')}, t._ingested_at) AS region_id,
           max_by({v('tower_id')}, t._ingested_at) AS tower_id
    FROM {B}.bronze_network_topology_master t GROUP BY 1""")
spark.sql(f"""CREATE OR REPLACE TEMP VIEW v_region_ref AS
    SELECT DISTINCT {v('region_id')} AS region_id FROM {B}.bronze_region_lookup t""")

bronze_pairs = [
    ("Topology cell region = tower master region", "bronze_network_topology_master", v("region_id"), "r.region_id",
     f"LEFT JOIN v_tower_ref r ON r.tower_id = {v('tower_id')}"),
    ("KPI region = tower master region", "bronze_network_kpi_measurements", v("region_id"), "r.region_id",
     f"LEFT JOIN v_tower_ref r ON r.tower_id = {v('tower_id')}"),
    ("KPI region = cell (topology) region", "bronze_network_kpi_measurements", v("region_id"), "r.region_id",
     f"LEFT JOIN v_cell_ref r ON r.cell_id = {v('cell_id')}"),
    ("KPI tower = cell's tower (topology)", "bronze_network_kpi_measurements", v("tower_id"), "r.tower_id",
     f"LEFT JOIN v_cell_ref r ON r.cell_id = {v('cell_id')}"),
    ("Complaint region = tower master region", "bronze_customer_complaints", v("region_id"), "r.region_id",
     f"LEFT JOIN v_tower_ref r ON r.tower_id = {v('tower_id')}"),
]
for name, t, cmp_expr, ref_expr, join_sql in bronze_pairs:
    r = q(f"SELECT COUNT_IF({cmp_expr} <> {ref_expr}) AS mismatched FROM {B}.{t} t {join_sql}")
    check("BRONZE", "REGION_CONSISTENCY", f"{name} (mismatched rows)", t, 0, r.mismatched, blocking=False)

for t in ["bronze_tower_master", "bronze_network_topology_master", "bronze_network_kpi_measurements", "bronze_customer_complaints"]:
    r = q(f"""SELECT COUNT_IF({v('region_id')} IS NOT NULL AND g.region_id IS NULL) AS mismatched
              FROM {B}.{t} t LEFT JOIN v_region_ref g ON g.region_id = {v('region_id')}""")
    check("BRONZE", "REGION_CONSISTENCY", "region_id exists in region lookup (rows not found)", t, 0, r.mismatched, blocking=False)

r = q("""SELECT COUNT_IF(n > 1) AS mismatched FROM
         (SELECT tower_id, COUNT(DISTINCT region_id) AS n FROM v_cell_ref WHERE region_id IS NOT NULL GROUP BY tower_id)""")
check("BRONZE", "REGION_CONSISTENCY", "All cells of a tower share one region (towers in >1 region)",
      "bronze_network_topology_master", 0, r.mismatched, blocking=False)
print(f"Bronze checks: {sum(1 for x in results if x[0] == 'BRONZE')}")

In [0]:
# 3. Silver checks (moved from 04_Silver_Processing: validation summary, rule reconciliation, region checks)
rej = spark.table(f"{S}.silver_rejected_records")
warn = spark.table(f"{S}.silver_warning_records")

rejected_records = {r.source_table: r.n for r in
                    rej.groupBy("source_table").agg(F.countDistinct("source_record_json").alias("n")).collect()}

for bt, (st, key) in DATASETS.items():
    n_bronze = spark.table(f"{B}.{bt}").count()
    r = q(f"SELECT COUNT(*) AS n, COUNT(DISTINCT {key}) AS keys, COUNT_IF({key} IS NULL) AS null_keys FROM {S}.{st}")
    check("SILVER", "RECONCILIATION", "Bronze rows = Silver rows + rejected records", st,
          n_bronze, r.n + rejected_records.get(bt, 0))
    check("SILVER", "UNIQUENESS", f"{key} unique in Silver (duplicate rows)", st, 0, r.n - r.keys)
    check("SILVER", "COMPLETENESS", f"{key} not null in Silver (rows)", st, 0, r.null_keys)
    # Warning flags on the Silver rows must match the warning table, pair by pair (key, check)
    flags = (spark.table(f"{S}.{st}")
             .select(F.col(key).cast("string").alias("k"), F.explode("dq_warning_checks").alias("c")).distinct())
    stored = (warn.filter(F.col("target_table") == st)
              .select(F.col("business_key_value").cast("string").alias("k"), F.col("warning_reason").alias("c")).distinct())
    check("SILVER", "WARNINGS", "dq_warning_checks = silver_warning_records (pairs only on one side)", st,
          0, flags.exceptAll(stored).count() + stored.exceptAll(flags).count())

# Rule catalogue and rule reconciliation: Bronze validation query = issues detected by the engine = stored records
cat = spark.table(f"{S}.silver_dq_rule_catalogue")
rule_results = spark.table(f"{S}.silver_dq_rule_results")
cat_rows = cat.select("rule_id", "validation_query").collect()
bronze_counts = spark.sql("\nUNION ALL\n".join(
    f"SELECT '{r.rule_id}' AS rule_id, ({r.validation_query}) AS bronze_violations" for r in cat_rows))
stored_rej = rej.groupBy("rule_id").agg(F.count("*").alias("stored_rejected"))
stored_warn = warn.groupBy("rule_id").agg(F.count("*").alias("stored_warnings"))

rule_recon = (cat.select("rule_id", "source_table", "rule_category", "check_name", "column_name", "severity", "issue_level")
    .join(bronze_counts, "rule_id", "left")
    .join(rule_results.select("rule_id", "detected_issues", "detected_on_valid_records"), "rule_id", "left")
    .join(stored_rej, "rule_id", "left")
    .join(stored_warn, "rule_id", "left")
    .fillna(0, ["bronze_violations", "detected_issues", "detected_on_valid_records", "stored_rejected", "stored_warnings"])
    # REJECT issues are stored for every rejected record; WARN issues are stored only for records kept in Silver
    .withColumn("expected_stored", F.when(F.col("issue_level") == "REJECT", F.col("detected_issues"))
                                    .otherwise(F.col("detected_on_valid_records")))
    .withColumn("actual_stored", F.when(F.col("issue_level") == "REJECT", F.col("stored_rejected"))
                                  .otherwise(F.col("stored_warnings"))))
recon = rule_recon.agg(
    F.count("*").alias("rules"), F.countDistinct("rule_id").alias("rule_ids"),
    F.count_if(F.col("bronze_violations") != F.col("detected_issues")).alias("bronze_vs_detected"),
    F.count_if(F.col("expected_stored") != F.col("actual_stored")).alias("detected_vs_stored")).first()

check("SILVER", "RULE_CATALOGUE", "Rule catalogue is not empty", "silver_dq_rule_catalogue", 1, recon.rules, ">=")
check("SILVER", "RULE_CATALOGUE", "rule_id unique in catalogue (duplicates)", "silver_dq_rule_catalogue", 0, recon.rules - recon.rule_ids)
check("SILVER", "RULE_CATALOGUE", "Every catalogue rule has a result for this run (rules missing)", "silver_dq_rule_results",
      0, cat.join(rule_results, "rule_id", "left_anti").count())
for name, df in [("silver_rejected_records", rej), ("silver_warning_records", warn)]:
    check("SILVER", "RULE_CATALOGUE", "Every stored record maps to a catalogue rule (records unmapped)", name,
          0, df.join(cat.select("rule_id"), "rule_id", "left_anti").count())
check("SILVER", "RULE_RECONCILIATION", "Bronze validation query = issues detected by the engine (rules differing)",
      "silver_dq_rule_catalogue", 0, recon.bronze_vs_detected)
check("SILVER", "RULE_RECONCILIATION", "Detected issues = stored rejected / warning records (rules differing)",
      "silver_dq_rule_catalogue", 0, recon.detected_vs_stored)

# Region consistency on the Silver outputs (non-blocking: source-data issue)
TWR, TOP, REG = f"{S}.silver_tower_master", f"{S}.silver_network_topology_master", f"{S}.silver_region_lookup"
KPI_T, CMP_T = f"{S}.silver_network_kpi_measurements", f"{S}.silver_customer_complaints"
silver_pairs = [
    ("Topology cell region = tower master region", TOP, "t.region_id", "r.region_id", f"LEFT JOIN {TWR} r ON r.tower_id = t.tower_id"),
    ("KPI region = tower master region", KPI_T, "t.region_id", "r.region_id", f"LEFT JOIN {TWR} r ON r.tower_id = t.tower_id"),
    ("KPI region = cell (topology) region", KPI_T, "t.region_id", "r.region_id", f"LEFT JOIN {TOP} r ON r.cell_id = t.cell_id"),
    ("KPI tower = cell's tower (topology)", KPI_T, "t.tower_id", "r.tower_id", f"LEFT JOIN {TOP} r ON r.cell_id = t.cell_id"),
    ("Complaint region = tower master region", CMP_T, "t.region_id", "r.region_id", f"LEFT JOIN {TWR} r ON r.tower_id = t.tower_id"),
]
for name, t, cmp_expr, ref_expr, join_sql in silver_pairs:
    r = q(f"SELECT COUNT_IF({cmp_expr} <> {ref_expr}) AS mismatched FROM {t} t {join_sql}")
    check("SILVER", "REGION_CONSISTENCY", f"{name} (mismatched rows)", t.split(".")[-1], 0, r.mismatched, blocking=False)
for t in (TWR, TOP, KPI_T, CMP_T):
    r = q(f"""SELECT COUNT_IF(t.region_id IS NOT NULL AND g.region_id IS NULL) AS mismatched
              FROM {t} t LEFT JOIN {REG} g ON g.region_id = t.region_id""")
    check("SILVER", "REGION_CONSISTENCY", "region_id exists in region lookup (rows not found)", t.split(".")[-1],
          0, r.mismatched, blocking=False)
r = q(f"""SELECT COUNT_IF(n > 1) AS mismatched FROM
          (SELECT tower_id, COUNT(DISTINCT region_id) AS n FROM {TOP} WHERE region_id IS NOT NULL GROUP BY tower_id)""")
check("SILVER", "REGION_CONSISTENCY", "All cells of a tower share one region (towers in >1 region)",
      "silver_network_topology_master", 0, r.mismatched, blocking=False)
print(f"Silver checks: {sum(1 for x in results if x[0] == 'SILVER')}")

# Rules that found something or disagree (detail for investigation)
display(rule_recon.filter("detected_issues > 0 OR bronze_violations > 0 OR expected_stored <> actual_stored").orderBy("rule_id"))


In [0]:
# 4. Gold checks (moved from 05_Gold_Analytics: validation report, handbook model validation, Gold region checks)
gt = lambda t: spark.table(f"{G}.{t}")
st = lambda t: spark.table(f"{S}.{t}")

GOLD_TABLES = ["dim_tower", "dim_region", "dim_cell", "fact_network_kpi", "fact_network_outage",
               "fact_customer_complaint", "fact_outage_impact", "gold_network_quality_daily", "gold_region_sla_summary"]
for t in GOLD_TABLES:
    check("GOLD", "VOLUME", "Table is not empty", t, 1, gt(t).count(), ">=")

d_tower, d_region, d_cell = gt("dim_tower"), gt("dim_region"), gt("dim_cell")
f_kpi, f_out, f_cmp, f_imp = gt("fact_network_kpi"), gt("fact_network_outage"), gt("fact_customer_complaint"), gt("fact_outage_impact")
daily, sla = gt("gold_network_quality_daily"), gt("gold_region_sla_summary")
silver_kpi = st("silver_network_kpi_measurements")

kpi_n, out_n, cmp_n = f_kpi.count(), f_out.count(), f_cmp.count()
outlier_n = f_kpi.filter("is_iqr_outlier").count()

# --- Facts vs Silver, grain and totals ---
check("GOLD", "RECONCILIATION", "fact_network_kpi = Silver KPI rows", "fact_network_kpi", silver_kpi.count(), kpi_n)
check("GOLD", "RECONCILIATION", "IQR_OUTLIER records kept in Gold", "fact_network_kpi",
      silver_kpi.filter(F.array_contains("dq_warning_checks", "IQR_OUTLIER")).count(), outlier_n)
check("GOLD", "RECONCILIATION", "fact_network_outage = Silver outages", "fact_network_outage", st("silver_network_outage_events").count(), out_n)
check("GOLD", "RECONCILIATION", "fact_customer_complaint = Silver complaints", "fact_customer_complaint", st("silver_customer_complaints").count(), cmp_n)
flag_diff = (f_kpi.select("measurement_id", F.col("dq_warning_checks").alias("g"))
             .join(silver_kpi.select("measurement_id", F.col("dq_warning_checks").alias("s")), "measurement_id", "full")
             .filter(~F.col("g").eqNullSafe(F.col("s"))).count())
check("GOLD", "RECONCILIATION", "dq_warning_checks identical to Silver (rows differing)", "fact_network_kpi", 0, flag_diff)

# Records rejected in Silver for non-duplicate reasons must not reach Gold (duplicates share the key of the kept copy)
rejected_kpi_keys = (st("silver_rejected_records")
    .filter((F.col("target_table") == "silver_network_kpi_measurements") & (F.col("issue_category") != "DUPLICATE"))
    .select(F.col("business_key_value").alias("measurement_id")).distinct())
check("GOLD", "RECONCILIATION", "KPI records rejected in Silver (non-duplicate) absent from Gold", "fact_network_kpi",
      0, f_kpi.join(rejected_kpi_keys, "measurement_id", "left_semi").count())

check("GOLD", "GRAIN", "One row per measurement_id", "fact_network_kpi", kpi_n, f_kpi.select("measurement_id").distinct().count())
check("GOLD", "GRAIN", "One row per outage_id", "fact_network_outage", out_n, f_out.select("outage_id").distinct().count())
check("GOLD", "GRAIN", "One row per complaint_id", "fact_customer_complaint", cmp_n, f_cmp.select("complaint_id").distinct().count())
check("GOLD", "GRAIN", "One row per region_id + summary_date", "gold_network_quality_daily",
      daily.count(), daily.select("region_id", "summary_date").distinct().count())
check("GOLD", "TOTALS", "Daily measurement_count total = fact_network_kpi", "gold_network_quality_daily", kpi_n, total(daily, "measurement_count"))
check("GOLD", "TOTALS", "Daily outlier_measurement_count total = flagged KPI records", "gold_network_quality_daily", outlier_n, total(daily, "outlier_measurement_count"))
check("GOLD", "TOTALS", "Daily outage_count total = fact_network_outage", "gold_network_quality_daily", out_n, total(daily, "outage_count"))
check("GOLD", "TOTALS", "Daily complaint_count total = fact_customer_complaint", "gold_network_quality_daily", cmp_n, total(daily, "complaint_count"))

# Independent SQL recomputation of the primary daily latency (outliers excluded)
latency_diff = q(f"""
  WITH r AS (SELECT region_id, measurement_date AS summary_date,
                    AVG(CASE WHEN NOT array_contains(dq_warning_checks, 'IQR_OUTLIER') THEN latency_ms END) AS lat
             FROM {G}.fact_network_kpi GROUP BY ALL)
  SELECT COUNT(*) AS n FROM r JOIN {G}.gold_network_quality_daily d USING (region_id, summary_date)
  WHERE NOT (abs(r.lat - d.avg_latency_ms) < 1e-6 OR (r.lat IS NULL AND d.avg_latency_ms IS NULL))""").n
check("GOLD", "KPI_LOGIC", "avg_latency_ms = independent recomputation (region-days differing)", "gold_network_quality_daily", 0, latency_diff)
check("GOLD", "KPI_LOGIC", "Region-days with avg_latency_ms > 1000 ms", "gold_network_quality_daily", 0, daily.filter("avg_latency_ms > 1000").count())

# --- Handbook model: dimensions, outage impact, SLA ---
NON_BUSINESS_COLS = ["_rescued_data", "_source_system", "_source_feed", "_source_table", "_source_file",
                     "_ingested_at", "dq_warning_checks"]
for d, src, key in [(d_tower, "silver_tower_master", "tower_id"), (d_region, "silver_region_lookup", "region_id"),
                    (d_cell, "silver_network_topology_master", "cell_id")]:
    name = {"tower_id": "dim_tower", "region_id": "dim_region", "cell_id": "dim_cell"}[key]
    check("GOLD", "DIMENSIONS", f"{name} = Silver rows, unique {key}", name, st(src).count(), d.select(key).distinct().count())
    check("GOLD", "DIMENSIONS", "No lineage/DQ columns in dimension", name, 0, len([c for c in d.columns if c in NON_BUSINESS_COLS]))
check("GOLD", "DIMENSIONS", "dim_cell towers exist in dim_tower (orphans)", "dim_cell", 0, d_cell.join(d_tower, "tower_id", "left_anti").count())

W = int(f_imp.agg(F.max("impact_window_minutes")).first()[0] or 0)
check("GOLD", "OUTAGE_IMPACT", "Single impact window size used (distinct values)", "fact_outage_impact",
      1, f_imp.select("impact_window_minutes").distinct().count())
indep = q(f"""
  SELECT
    (SELECT COUNT(*) FROM {G}.fact_network_outage o JOIN {G}.fact_network_kpi k ON k.tower_id = o.tower_id
       AND k.measurement_ts BETWEEN o.outage_start_ts - INTERVAL {W} MINUTES AND o.outage_end_ts + INTERVAL {W} MINUTES) AS window_pairs,
    (SELECT COUNT(*) FROM {G}.fact_network_outage o JOIN {G}.fact_network_kpi k ON k.tower_id = o.tower_id
       AND k.measurement_ts BETWEEN o.outage_start_ts AND o.outage_end_ts) AS during_pairs,
    (SELECT COUNT(*) FROM {G}.fact_network_outage o JOIN {G}.fact_customer_complaint c ON c.tower_id = o.tower_id
       AND c.complaint_ts BETWEEN o.outage_start_ts - INTERVAL {W} MINUTES AND o.outage_end_ts + INTERVAL {W} MINUTES) AS complaint_pairs,
    (SELECT COUNT(DISTINCT k.measurement_id) FROM {G}.fact_network_outage o JOIN {G}.fact_network_kpi k ON k.tower_id = o.tower_id
       AND k.measurement_ts BETWEEN o.outage_start_ts - INTERVAL {W} MINUTES AND o.outage_end_ts + INTERVAL {W} MINUTES) AS windowed_measurements
""")
baseline_total = f_imp.select("tower_id", "baseline_measurement_count").distinct().agg(F.sum("baseline_measurement_count")).first()[0] or 0
other_tower_kpi = f_kpi.join(f_imp.select("tower_id").distinct(), "tower_id", "left_anti").count()   # towers without outages

check("GOLD", "OUTAGE_IMPACT", "One row per outage", "fact_outage_impact", out_n, f_imp.select("outage_id").distinct().count())
check("GOLD", "OUTAGE_IMPACT", "Rows = outages", "fact_outage_impact", out_n, f_imp.count())
check("GOLD", "OUTAGE_IMPACT", "Impact-window measurements = independent SQL", "fact_outage_impact", indep.window_pairs, total(f_imp, "window_measurement_count"))
check("GOLD", "OUTAGE_IMPACT", "During-outage measurements = independent SQL", "fact_outage_impact", indep.during_pairs, total(f_imp, "during_outage_measurement_count"))
check("GOLD", "OUTAGE_IMPACT", "Impact-window complaints = independent SQL", "fact_outage_impact", indep.complaint_pairs, total(f_imp, "window_complaint_count"))
check("GOLD", "OUTAGE_IMPACT", "Baseline + windowed (outage towers) + other towers = KPI fact", "fact_outage_impact",
      kpi_n, baseline_total + indep.windowed_measurements + other_tower_kpi)

check("GOLD", "SLA", "Daily rows without region attributes", "gold_network_quality_daily", 0, daily.filter(F.col("sla_tier").isNull()).count())
check("GOLD", "SLA", "One row per region", "gold_region_sla_summary", d_region.count(), sla.select("region_id").distinct().count())
check("GOLD", "SLA", "SLA measurement total = KPI fact", "gold_region_sla_summary", kpi_n, total(sla, "measurement_count"))
check("GOLD", "SLA", "SLA outage total = outage fact", "gold_region_sla_summary", out_n, total(sla, "outage_count"))
check("GOLD", "SLA", "SLA complaint total = complaint fact", "gold_region_sla_summary", cmp_n, total(sla, "complaint_count"))
check("GOLD", "SLA", "SLA outlier total = flagged KPI records", "gold_region_sla_summary", outlier_n, total(sla, "outlier_measurement_count"))

# --- Gold region logic (blocking: these test Gold's own joins / flags) and KPI-vs-tower region (non-blocking) ---
r = q(f"""SELECT COUNT_IF(t.region_id <> r.region_id) AS mismatched
          FROM {G}.fact_network_outage t LEFT JOIN {S}.silver_tower_master r ON r.tower_id = t.tower_id""")
check("GOLD", "REGION_LOGIC", "Outage region = tower master region (mismatched rows)", "fact_network_outage", 0, r.mismatched)
r = q(f"""SELECT COUNT_IF(NOT (f.kpi_tower_region_match <=> (s.region_id = r.region_id))) AS mismatched
          FROM {G}.fact_network_kpi f JOIN {S}.silver_network_kpi_measurements s USING (measurement_id)
          LEFT JOIN {S}.silver_tower_master r ON r.tower_id = s.tower_id""")
check("GOLD", "REGION_LOGIC", "kpi_tower_region_match = recomputed flag (rows differing)", "fact_network_kpi", 0, r.mismatched)
check("GOLD", "REGION_CONSISTENCY", "KPI region differs from tower master region (rows)", "fact_network_kpi",
      0, f_kpi.filter(~F.col("kpi_tower_region_match")).count(), blocking=False)
print(f"Gold checks: {sum(1 for x in results if x[0] == 'GOLD')}")

# Outage-to-telemetry correlation overview
display(f_imp.agg(
    F.count("*").alias("outages"),
    F.sum(F.col("has_telemetry_evidence").cast("int")).alias("outages_with_telemetry"),
    F.sum(F.col("has_customer_impact").cast("int")).alias("outages_with_complaints"),
    F.round(F.avg("latency_delta_ms"), 2).alias("avg_latency_delta_ms"),
    F.round(F.avg("availability_delta_pct"), 3).alias("avg_availability_delta_pct")))

In [0]:
# 5. Save one validation summary table per layer (overwritten each run: current state only, like the rejected/warning tables)
validated_at = spark.sql("SELECT current_timestamp() AS ts").first().ts
report = (spark.createDataFrame(results,
        "layer string, check_group string, check_name string, table_name string, comparison string, "
        "expected_value long, actual_value long, blocking boolean, status string")
    .withColumn("validated_at", F.lit(validated_at)))

for layer, target in SUMMARY_TABLES.items():
    (report.filter(F.col("layer") == layer).write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target))
    print(f"Saved {target}")

display(report.groupBy("layer").pivot("status", ["PASS", "WARN", "FAIL"]).count().fillna(0).orderBy("layer"))
display(report.orderBy(F.expr("CASE status WHEN 'FAIL' THEN 0 WHEN 'WARN' THEN 1 ELSE 2 END"), "layer", "check_group", "table_name"))

In [0]:
# 6. Fail the job task when a blocking check failed (summary tables are already saved for investigation)
failed = [r for r in results if r[-1] == "FAIL"]
warned = [r for r in results if r[-1] == "WARN"]
print(f"Checks: {len(results)} | PASS: {len(results) - len(failed) - len(warned)} | WARN: {len(warned)} | FAIL: {len(failed)}")
if failed and FAIL_ON_ERROR:
    raise AssertionError(f"{len(failed)} blocking validation checks failed: " +
                         "; ".join(f"{r[0]} {r[3]}: {r[2]} (expected {r[5]}, actual {r[6]})" for r in failed[:10]))